### Apartment Price Prediction in Morocco

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
from pathlib import Path

# Use the local file or the dataset attached to the notebook on Kaggle.
local_csv = Path('appartements_data.csv')
kaggle_csv = Path('/kaggle/input/appartements-data/appartements_data.csv')
csv_path = local_csv if local_csv.is_file() else kaggle_csv
if not csv_path.is_file():
    raise FileNotFoundError(
        "appartements_data.csv not found. Place it in the working directory "
        "or add the appartements-data dataset on Kaggle."
    )
print(f"Data loaded from: {csv_path}")

In [ ]:
import warnings
warnings.filterwarnings("ignore")


### **Importing the Data**

In [ ]:
appartements_data = pd.read_csv(csv_path)
appartements_data.head()

### Data Cleaning and Transformation

#### Converting apartment prices from object to numeric values

In [ ]:
print("Before starting the analysis, convert price strings to numeric values, "
      "as price is the target variable.\n")

# Remove nonnumeric characters and convert the values to floats
'''Convert the values to strings, remove nonnumeric characters while retaining decimal points,
then convert them to numbers. Empty or invalid values become missing values.
'''
appartements_data['price'] = appartements_data['price'].astype(str).str.replace(r'[^\d.]', '', regex=True)
appartements_data['price'] = pd.to_numeric(appartements_data['price'], errors='coerce')

# Display the resulting data type
print("\n Price data type after conversion:")
print(appartements_data['price'].dtype)


#### Extracting equipment as Boolean variables

In [ ]:
# Get the unique equipment values in the feature
all_equipments = appartements_data['equipment'].str.split('/').explode().unique()

# Create a column for each equipment item and add it to the dataset
for equipment in all_equipments:
    equipment_str = str(equipment)
    if equipment_str: # use False for missing values
        appartements_data[equipment_str.strip()] = appartements_data['equipment'].str.contains(equipment_str, na=False)

# Drop the equipment feature and the nan column
appartements_data.drop('equipment', axis=1, inplace=True)
appartements_data.drop('nan' , axis =1 , inplace= True)
# Display the transformed dataset
appartements_data.head()

#### Standardizing city names (Arabic → French)

In [ ]:
'''First list the cities in the dataset and find their French equivalents.'''
print(appartements_data['city_name'].unique())



In [ ]:

# Define a mapping for city names
city_mapping = {
    'الدار البيضاء': 'Casablanca',
    'الرباط': 'Rabat',
    'مراكش': 'Marrakech',
    'أكادير': 'Agadir',
    'القنيطرة' : 'Kénitra',
    'طنجة' : 'Tanger',
    'المحمدية' : 'Mohammedia'
}

# Apply the mapping to the city_name column
appartements_data['city_name'] = appartements_data['city_name'].replace(city_mapping)

# Display the result
print("Unique city names after unification:")
print(appartements_data['city_name'].unique())

In [ ]:
### Inspect the dataset
# The price type has been adjusted and can be used
# in the exploratory analysis

appartements_data.info()


In [ ]:
# Inspect summary statistics
appartements_data.describe()

## **Exploratory Data Analysis**

#### Univariate analysis

In [ ]:
appartements_data.hist(figsize=(20,15),  grid = False )

In [ ]:
# Remove nan from all_equipments before dropping columns
all_equipments_filtered = [eq for eq in all_equipments if pd.notna(eq)]

appartements_data.drop(columns=all_equipments_filtered).boxplot( figsize = (15, 9),grid=False, patch_artist=True,
                    flierprops=dict(marker='o', markersize=6))

'''Price values are widely dispersed and include outliers, making outliers in other features difficult to see on this plot.'''

In [ ]:
# Plot each numeric feature separately to inspect outliers
for col in appartements_data.select_dtypes(include = np.number).columns:
    appartements_data.boxplot(column=col, figsize=(6,4), grid=False, patch_artist=True,
                   flierprops=dict(marker='o', markersize=6))
    plt.title(col)
    plt.show()


In [ ]:
df_num = appartements_data.select_dtypes(include=np.number)

# Apply log(1+x) to reduce the spread
df_log = np.log1p(df_num)

df_log.boxplot(
    figsize=(20,9),
    grid=False,
    patch_artist=True,
    flierprops=dict(marker='o', markersize=6)
)

'''Reducing the spread makes the boxplots easier to interpret.'''

#### Bivariate analysis

In [ ]:
## Price variation
from matplotlib import pyplot as plt
appartements_data['price'].plot(kind='line', figsize=(10, 7), title='price')
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
sns.pairplot(appartements_data.drop(columns=all_equipments_filtered) , hue = 'price')
'''The visualization suggests that most features have little relationship with one another.'''

In [ ]:

correlation_matrix = appartements_data.select_dtypes(include=["number", "bool"]).corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation matrix for numeric values')
plt.show()
'''The correlation matrix suggests that most features are not strongly correlated.
The number of rooms and surface area are correlated, while price and surface area are weakly correlated.
For this analysis, select nonredundant features with an absolute correlation above 0.15 with price.
'''

# Select features correlated with the target price

threshold = 0.15
# Exclude price itself and retain correlations above the threshold
high_corr_features = correlation_matrix['price'][correlation_matrix['price'].abs() > threshold].index.tolist()

# Remove price from the list
high_corr_features = [feat for feat in high_corr_features if feat != 'price']

print("Features strongly correlated with price (>0.15):")
print(high_corr_features)


# Create a correlation matrix for the selected features
subset_corr = correlation_matrix.loc[high_corr_features, high_corr_features].abs()

# Final list of nonredundant features
selected_features = []

for col in subset_corr.columns:
    # Check whether this column is strongly correlated with an already selected feature
    if not any(subset_corr[col][selected_features] > 0.8):
        selected_features.append(col)

print("\nFinal nonredundant features:")
print(selected_features)

In [ ]:
## Collect variables not selected by correlation with price
non_selected_features = [feat for feat in appartements_data.columns if feat not in selected_features]
print(non_selected_features)

## Handling Outliers and Missing Values

### Missing values

In [ ]:
# Create a heatmap to detect missing feature values
plt.figure(figsize=(10,6))
sns.heatmap(appartements_data.isnull(), cbar=False, cmap="viridis", yticklabels=False)
plt.title("Missing values in the dataset")
plt.show()

In [ ]:
appartements_data.isnull().sum()

In [ ]:
'''Some features have missing values. Numeric values will be replaced with the median.
Categorical values such as title and city_name can be handled separately, for example with the mode or an unknown value.
'''
# Select only numerical columns
numerical_cols = appartements_data.select_dtypes(include=np.number).columns

# Fill missing values in numerical columns with the median of each column
appartements_data[numerical_cols] = appartements_data[numerical_cols].fillna(appartements_data[numerical_cols].median())

# You might want to handle missing values in non-numerical columns separately if needed.
# For example, for 'title' and 'city_name', which have only one missing value each:
appartements_data['title'].fillna('Inconnu', inplace=True)
appartements_data['city_name'].fillna('Inconnu', inplace=True)


print("\nNumber of missing values after imputation:")
print(appartements_data.isnull().sum())

### Handling outliers

In [ ]:
'''The boxplots show outliers in the target. For apartment prices, which are often skewed and contain extreme values, the IQR approach is considered more robust than a Z-score approach.
'''
print(f"\n Number of rows before removing outliers: {appartements_data.shape[0]}")
# Keep numeric features
numerical_features = appartements_data.select_dtypes(include=np.number).columns

# Copy the dataset for cleaning
df_cleaned = appartements_data.copy()

for feature in numerical_features:
    # Calculate Q1, Q3, and IQR
    Q1 = df_cleaned[feature].quantile(0.25)
    Q3 = df_cleaned[feature].quantile(0.75)
    IQR = Q3 - Q1

    # Define the bounds
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Count outliers before removal
    outliers_count = df_cleaned[(df_cleaned[feature] < lower_bound) | (df_cleaned[feature] > upper_bound)].shape[0]
    print(f"Feature '{feature}' : {outliers_count} outliers detected")

    # Remove detected outliers
    df_cleaned = df_cleaned[(df_cleaned[feature] >= lower_bound) & (df_cleaned[feature] <= upper_bound)]

     # Boxplot before and after
    plt.figure(figsize=(6,4))
    plt.boxplot([appartements_data[feature], df_cleaned[feature]], labels=['Before', 'After'])
    plt.title(f"Outliers for '{feature}'")
    plt.show()

print(f"\nCleaned dataset: {df_cleaned.shape[0]} rows remaining after outlier removal")



## Encoding Categorical and Boolean Variables

Encode categorical and Boolean variables so the models can interpret them.

In [ ]:
# Encode Boolean values
bool_columns = appartements_data.select_dtypes(include='bool').columns.tolist()
df_cleaned[bool_columns] = df_cleaned[bool_columns].astype(int)

df_cleaned.head()

#### Label encoding for selected machine learning models

In [ ]:
from sklearn.preprocessing import LabelEncoder
'''Use label encoding for decision trees, random forests, and gradient boosting.
'''
data_label_encoding = df_cleaned.copy()
# Encode categorical variables
le = LabelEncoder()
# Apply the encoding
data_label_encoding['city_name'] = le.fit_transform(data_label_encoding['city_name'])
data_label_encoding.head()

#### One-hot encoding for selected machine learning models

In [ ]:
# One-hot encode cities
'''This approach is suitable for linear regression, SVM, and KNN.
'''

# Standardize capitalization
data_onehot_encoding = df_cleaned.copy()
data_onehot_encoding['city_name'] = data_onehot_encoding['city_name'].str.title()

data_onehot_encoding= pd.get_dummies(data_onehot_encoding, columns=['city_name'])
data_onehot_encoding.head()

### Scaling numerical values

In [ ]:
from sklearn.preprocessing import StandardScaler

se = StandardScaler()
data_onehot_encoding[numerical_features] = se.fit_transform(data_onehot_encoding[numerical_features])
data_onehot_encoding.head()



In [ ]:
data_label_encoding[numerical_features] = se.fit_transform(data_label_encoding[numerical_features])
data_label_encoding.head()

## Selecting explanatory variables

In [ ]:
print(selected_features)
print(non_selected_features)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

'''Remove title and link, which are considered of limited value for predicting apartment prices.
'''
#data_onehot_encoding = data_onehot_encoding.drop([], axis=1)
#data_label_encoding = data_label_encoding.drop(['title', 'link'], axis=1)


'''Features with a correlation below 0.15 with the target are considered insignificant for this analysis.
'''


# Preserve the city indicator columns
non_selected_features.remove("city_name")
print(non_selected_features)
X_onehot = data_onehot_encoding.drop(non_selected_features, axis=1)
print(X_onehot.columns)

y_onehot = data_onehot_encoding['price']




### Keep only significant features
selected_features = selected_features + ['city_name']
print(selected_features)
X_label = data_label_encoding.drop(['price' , 'title', 'link'], axis=1)
X_label = X_label[selected_features]
y_label = data_label_encoding['price']





## Splitting the Data

In [ ]:
# Split the data into training and test sets

X_train_onehot, X_test_onehot, y_train_onehot, y_test_onehot = train_test_split(X_onehot, y_onehot, test_size=0.2, random_state=42)
X_train_label, X_test_label, y_train_label, y_test_label = train_test_split(X_label, y_label, test_size=0.2, random_state=42)


## Training Machine Learning Models

## One-Hot Encoded Dataset

### KNN

One-hot encoding of cities is suitable for linear regression, SVM, and KNN.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score , accuracy_score, classification_report
knn = KNeighborsRegressor(
    n_neighbors=6,     # number of neighbors
    weights="distance", # distance weighting
    metric="minkowski", # Minkowski distance
    p=2,                # p=2 -> Euclidean, p=1 -> Manhattan
    leaf_size=30,       # parameter for optimizing neighbor search
    n_jobs=-1           # use all CPU cores
)
knn.fit(X_train_onehot, y_train_onehot)

y_pred_onehot = knn.predict(X_test_onehot)


mse_onehot = mean_squared_error(y_test_onehot, y_pred_onehot)
r2_onehot = r2_score(y_test_onehot, y_pred_onehot)

print(f"Mean Squared Error (One-Hot Encoding): {mse_onehot:.2f}")
print(f"R-squared (One-Hot Encoding): {r2_onehot:.2f}\n")

valeur_comparaison = pd.DataFrame({'Actual value': y_test_onehot, 'Predicted value': y_pred_onehot})
print(valeur_comparaison)

### SVM

In [ ]:
from sklearn.svm import SVR

svm = SVR(kernel='rbf', C=1.0, epsilon=0.1)
svm.fit(X_train_onehot, y_train_onehot)

y_pred_onehot = svm.predict(X_test_onehot)

mse_onehot = mean_squared_error(y_test_onehot, y_pred_onehot)
r2_onehot = r2_score(y_test_onehot, y_pred_onehot)

print(f"Mean Squared Error (One-Hot Encoding): {mse_onehot:.2f}")
print(f"R-squared (One-Hot Encoding): {r2_onehot:.2f}\n")

valeur_comparaison = pd.DataFrame({'Actual value': y_test_onehot, 'Predicted value': y_pred_onehot})
print(valeur_comparaison)


#### Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

lr = LinearRegression()
lr.fit(X_train_onehot, y_train_onehot)

y_pred_onehot = lr.predict(X_test_onehot)

mae_label = mean_squared_error(y_test_onehot, y_pred_onehot)
mse_label = mean_squared_error(y_test_onehot, y_pred_onehot)
r2_label = r2_score(y_test_onehot, y_pred_onehot)

print(f"Mean Squared Error (Label Encoding): {mse_label:.2f}")
print(f"R-squared (Label Encoding): {r2_label:.2f}\n")

valeur_comparaison = pd.DataFrame({'Actual value': y_test_onehot, 'Predicted value': y_pred_onehot})
print(valeur_comparaison)


In [ ]:
from sklearn.linear_model import  Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Models to test
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01),
    "ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5)
}

# Store the results
results = []

# Iterate over the models
for name, model in models.items():
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        (name, model)
    ])

    # Training
    pipe.fit(X_train_onehot, y_train_onehot)

    # Predictions
    y_pred = pipe.predict(X_test_onehot)

    # Metrics
    mae = mean_absolute_error(y_test_onehot, y_pred)
    mse = mean_squared_error(y_test_onehot, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test_onehot, y_pred)

    results.append([name, mae, rmse, r2])

# Results in a DataFrame
results_df = pd.DataFrame(results, columns=["Model", "MAE", "RMSE", "R²"])
print(results_df)

## Label Encoded Dataset

### KNN

In [ ]:
knn = KNeighborsRegressor(
    n_neighbors=6,     # number of neighbors
    weights="distance", # distance weighting
    metric="minkowski", # Minkowski distance
    p=2,                # p=2 -> Euclidean, p=1 -> Manhattan
    leaf_size=30,       # parameter for optimizing neighbor search
    n_jobs=-1           # use all CPU cores
)
knn.fit(X_train_label, y_train_label)

y_pred_label = knn.predict(X_test_label)


mse_label = mean_squared_error(y_test_label, y_pred_label)
r2_label = r2_score(y_test_label, y_pred_label)

print(f"Mean Squared Error (Label Encoding): {mse_label:.2f}")
print(f"R-squared (Label Encoding): {r2_label:.2f}\n")

valeur_comparaison = pd.DataFrame({'Actual value': y_test_label, 'Predicted value': y_pred_label})
print(valeur_comparaison)

### SVM

In [ ]:
svm = SVR(kernel='rbf', C=1.0, epsilon=0.1)
svm.fit(X_train_label, y_train_label)

y_pred_label = svm.predict(X_test_label)

mse_onehot = mean_squared_error(y_test_label, y_pred_label)
r2_onehot = r2_score(y_test_label, y_pred_label)

print(f"Mean Squared Error (One-Hot Encoding): {mse_label:.2f}")
print(f"R-squared (One-Hot Encoding): {r2_label:.2f}\n")

valeur_comparaison = pd.DataFrame({'Actual value': y_test_label, 'Predicted value': y_pred_label})
print(valeur_comparaison)


### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rd = RandomForestRegressor(n_estimators=100, random_state=42 ,max_depth =10)
rd.fit(X_train_label, y_train_label)

y_pred_label = rd.predict(X_test_label)

mse_label = mean_squared_error(y_test_label, y_pred_label)
r2_label = r2_score(y_test_label, y_pred_label)

print(f"Mean Squared Error (Label Encoding): {mse_label:.2f}")
print(f"R-squared (Label Encoding): {r2_label:.2f}\n")

valeur_comparaison = pd.DataFrame({'Actual value': y_test_label, 'Predicted value': y_pred_label})
print(valeur_comparaison)


### Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42 , max_depth=3)


gb.fit(X_train_label, y_train_label)
y_pred_label = gb.predict(X_test_label)

mse_label = mean_squared_error(y_test_label, y_pred_label)
r2_label = r2_score(y_test_label, y_pred_label)

print(f"Mean Squared Error (Label Encoding): {mse_label :.2f}")
print(f"R-squared (Label Encoding): {r2_label :.2f}\n")

valeur_comparaison = pd.DataFrame({'Actual value': y_test_label, 'Predicted value': y_pred_label})
print(valeur_comparaison)


### Neural Network (MLP Regressor)

In [ ]:
! pip install tensorflow

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dropout

model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(X_train_label.shape[1],)))
model.add(LeakyReLU(alpha=0.1))
model.add(Dropout(0.2))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='linear'))


optimizer = Adam(learning_rate=0.001)
model.compile(optimizer = optimizer , loss='mean_squared_error')

# early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model.fit(X_train_label, y_train_label, epochs=100, validation_split=0.2,
          #callbacks=[early_stop],
          batch_size=32, verbose=1)

y_pred_label = model.predict(X_test_label)

mse_label = mean_squared_error(y_test_label, y_pred_label)
r2_label = r2_score(y_test_label, y_pred_label)

print(f"Mean Squared Error (Label Encoding): {mse_label:.2f}")
print(f"R-squared (Label Encoding): {r2_label:.2f}\n")

valeur_comparaison = pd.DataFrame({'Actual value': y_test_label, 'Predicted value': y_pred_label.flatten()})
print(valeur_comparaison)

### Regression Model Summary

Based on the R² scores obtained after training several models with one-hot and label encoding, the recorded results are:

* **KNN**: one-hot R² = 0.36; label encoding R² = 0.37
* **SVM**: one-hot R² = 0.31; label encoding R² = 0.37
* **Linear models** (Linear Regression, Ridge, Lasso, ElasticNet), with one-hot encoding: R² ≈ 0.20–0.21
* **Random Forest**, with label encoding: R² = 0.31
* **Gradient Boosting**, with label encoding: R² = 0.35
* **Neural network**, with label encoding: R² = 0.23

**Most suitable model among those tested**

Based on R², which measures the proportion of price variance explained by the model, **KNN** and **SVM** perform best, with an R² of approximately **0.37** when `city_name` is label encoded.

This suggests these models are the most suitable for predicting apartment prices *at this stage of the analysis*. However, an R² of 0.37 means that only about 37% of the variance in prices is explained, leaving considerable room for improvement.

**Potential improvements**

* Tune the hyperparameters of KNN and SVM.
* Explore more advanced models, such as XGBoost or LightGBM.
* Analyze the features in more depth and engineer more relevant features.
* Refine the handling of missing values and outliers.

## Project Summary and Conclusion

### Summary

This project explored apartment data from Morocco and prepared it for modeling. The main steps were:

- Importing and inspecting the dataset.
- Converting the price column to numeric values.
- Extracting and encoding equipment as Boolean variables.
- Standardizing city names.
- Handling missing values through mode imputation.
- Identifying and handling outliers in numeric features, especially price and surface area.
- Selecting the features most correlated with price.
- Encoding categorical variables (cities) using label encoding and one-hot encoding.
- Scaling numeric features.
- Splitting the data into training and test sets.
- Training several regression models (KNN, SVM, Linear Regression, Ridge, Lasso, ElasticNet, Random Forest, Gradient Boosting, and a simple neural network) on differently encoded datasets.

Model performance was evaluated using mean squared error (MSE) and R-squared (R²). The results obtained so far vary by model and categorical encoding method.

### Conclusion

The initial analysis and modeling provided insights into factors influencing apartment prices in Morocco. Although some models showed moderate predictive ability (R² around 0.30–0.38 for the best models tested), there is significant room for improvement. The low R² values suggest that the current models explain only part of the variance in apartment prices.

Outliers, the data distribution even after transformation, and the inherent complexity of the real estate market may be limiting factors. Handling missing values with the mode, while simple, may not be optimal for every feature.

### Possible Improvements

1. **Advanced missing-value handling:** Experiment with other imputation techniques, such as model-based or multivariate imputation, or investigate why values are missing.
2. **Deeper outlier analysis and handling:** Investigate remaining outliers and consider more robust or feature-specific detection and treatment methods.
3. **More extensive feature engineering:** Create relevant features, for example by extracting neighborhood information from listing titles or combining existing features such as surface area per room.
4. **Explore other models:** Test other regression algorithms, such as XGBoost, LightGBM, or more sophisticated tree-based models.
5. **Advanced hyperparameter tuning:** Use grid search, random search, or Bayesian optimization to find better settings for selected models.
6. **Robust cross-validation:** Use more advanced cross-validation schemes, such as stratified K-fold if prices can be grouped into bins, for a more reliable evaluation.
7. **Collect additional data:** If possible, enrich the dataset with other sources, such as socioeconomic information by city or neighborhood, nearby amenities, and construction date.
8. **Model error analysis:** Examine predictions with the largest errors to identify patterns and potentially problematic or missing features.